In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings('ignore')
import joblib
from config.paths import config
from src.data_loader import data_loader

config.create_directories()

try:
    df = pd.read_csv(config.processed_data_dir / "cleaned_data.csv")
    print(f"Loaded cleaned data: {df.shape}")
except:
    print("Loading raw data...")
    df = data_loader.load_non_medical_data()

try:
    feature_classification = pd.read_csv(config.reports_dir / "feature_classification.csv")
    print("Loaded feature classification")
    print(f"Columns in feature classification: {feature_classification.columns.tolist()}")
except:
    print("Feature classification not available")
    feature_classification = None

print(f"Original data shape: {df.shape}")

Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\data\raw
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\data\processed
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\data\external
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\models
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\visuals\eda
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\visuals\feature_importance
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\visuals\model_performance
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\reports
Created: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindSc

In [2]:
medical_features = []
if feature_classification is not None:
    medical_features = feature_classification[feature_classification['Category'] == 'Medical']['Feature'].tolist()

features_to_drop = medical_features + ['HISPOR', 'RACESEC', 'RACETER', 'INHISPOR', 'INRASEC', 'INRATER']
features_to_drop = [f for f in features_to_drop if f in df.columns and f != 'DEMENTED']

print(f"Dropping {len(features_to_drop)} features: {features_to_drop}")
df_engineered = df.drop(columns=features_to_drop)

high_corr_features = ['INRACE', 'INEDUC']
df_engineered = df_engineered.drop(columns=[f for f in high_corr_features if f in df_engineered.columns and f != 'DEMENTED'])

numerical_features = df_engineered.select_dtypes(include=[np.number]).columns
numerical_features = [f for f in numerical_features if f != 'DEMENTED']

if len(numerical_features) > 0:
    X_numerical = df_engineered[numerical_features].fillna(df_engineered[numerical_features].median())
    selector = VarianceThreshold(threshold=0.05 * (1 - 0.05))
    selector.fit(X_numerical)
    low_variance_features = [numerical_features[i] for i in range(len(numerical_features)) if not selector.get_support()[i]]
    df_engineered = df_engineered.drop(columns=low_variance_features)
    print(f"Dropped {len(low_variance_features)} low variance features")

print(f"After feature selection: {df_engineered.shape}")

Dropping 6 features: ['HISPOR', 'RACESEC', 'RACETER', 'INHISPOR', 'INRASEC', 'INRATER']
Dropped 5 low variance features
After feature selection: (195196, 44)


In [3]:
numerical_features = df_engineered.select_dtypes(include=[np.number]).columns
numerical_features = [f for f in numerical_features if f != 'DEMENTED']

categorical_features = df_engineered.select_dtypes(include=['object']).columns

for col in numerical_features:
    if df_engineered[col].isnull().sum() > 0:
        df_engineered[f'{col}_missing'] = df_engineered[col].isnull().astype(int)
        df_engineered[col] = df_engineered[col].fillna(df_engineered[col].median())
        print(f"Imputed {col} with median + missing indicator")

for col in categorical_features:
    if df_engineered[col].isnull().sum() > 0:
        df_engineered[col] = df_engineered[col].fillna('Unknown')
    df_engineered[col] = df_engineered[col].astype(str)
    print(f"Processed categorical: {col}")

print(f"After missing data handling: {df_engineered.shape}")

Imputed VISITMO with median + missing indicator
Imputed VISITDAY with median + missing indicator
Imputed NACCVNUM with median + missing indicator
Imputed NACCAVST with median + missing indicator
Imputed NACCNVST with median + missing indicator
Imputed NACCDAYS with median + missing indicator
Imputed NACCFDYS with median + missing indicator
Imputed BIRTHMO with median + missing indicator
Imputed HISPANIC with median + missing indicator
Imputed RACE with median + missing indicator
Imputed PRIMLANG with median + missing indicator
Imputed EDUC with median + missing indicator
Imputed MARISTAT with median + missing indicator
Imputed NACCLIVS with median + missing indicator
Imputed INDEPEND with median + missing indicator
Imputed RESIDENC with median + missing indicator
Imputed HANDED with median + missing indicator
Imputed INBIRMO with median + missing indicator
Imputed INBIRYR with median + missing indicator
Imputed INSEX with median + missing indicator
Imputed NACCNINR with median + missin

In [4]:
if all(f in df_engineered.columns for f in ['INDEPEND', 'RESIDENC', 'NACCLIVS']):
    lifestyle_weights = {'INDEPEND': 0.4, 'RESIDENC': 0.3, 'NACCLIVS': 0.3}
    df_engineered['healthy_lifestyle_score'] = (
        df_engineered['INDEPEND'] * lifestyle_weights['INDEPEND'] +
        df_engineered['RESIDENC'] * lifestyle_weights['RESIDENC'] +
        df_engineered['NACCLIVS'] * lifestyle_weights['NACCLIVS']
    )
    print("Created healthy_lifestyle_score")

if all(f in df_engineered.columns for f in ['NACCLIVS', 'INVISITS', 'INCALLS']):
    social_weights = {'NACCLIVS': 0.5, 'INVISITS': 0.3, 'INCALLS': 0.2}
    df_engineered['social_engagement_index'] = (
        df_engineered['NACCLIVS'] * social_weights['NACCLIVS'] +
        df_engineered['INVISITS'] * social_weights['INVISITS'] +
        df_engineered['INCALLS'] * social_weights['INCALLS']
    )
    print("Created social_engagement_index")

if 'EDUC' in df_engineered.columns and 'NACCLIVS' in df_engineered.columns:
    df_engineered['education_social_interaction'] = df_engineered['EDUC'] * df_engineered['NACCLIVS']
    print("Created education_social_interaction")

if 'NACCAGE' in df_engineered.columns:
    age_bins = [0, 65, 75, 85, 120]
    age_labels = ['Younger', 'Middle', 'Senior', 'Elderly']
    df_engineered['age_group'] = pd.cut(df_engineered['NACCAGE'], bins=age_bins, labels=age_labels)
    print("Created age_group")

if 'EDUC' in df_engineered.columns:
    educ_bins = [0, 12, 16, 20]
    educ_labels = ['Basic', 'Intermediate', 'Advanced']
    df_engineered['education_level'] = pd.cut(df_engineered['EDUC'], bins=educ_bins, labels=educ_labels)
    print("Created education_level")

numerical_for_poly = ['NACCAGE', 'EDUC', 'INDEPEND']
for col in numerical_for_poly:
    if col in df_engineered.columns:
        df_engineered[f'{col}_squared'] = df_engineered[col] ** 2
        print(f"Created {col}_squared")

print(f"After feature creation: {df_engineered.shape}")

Created healthy_lifestyle_score
Created social_engagement_index
Created education_social_interaction
Created age_group
Created education_level
Created NACCAGE_squared
Created EDUC_squared
Created INDEPEND_squared
After feature creation: (195196, 88)


In [5]:
categorical_features = df_engineered.select_dtypes(include=['object', 'category']).columns

low_cardinality_features = []
high_cardinality_features = []

for col in categorical_features:
    if df_engineered[col].nunique() <= 10:
        low_cardinality_features.append(col)
    else:
        high_cardinality_features.append(col)

print(f"Low cardinality features: {len(low_cardinality_features)}")
print(f"High cardinality features: {len(high_cardinality_features)}")

df_encoded = pd.get_dummies(df_engineered, columns=low_cardinality_features, drop_first=True)

for col in high_cardinality_features:
    if col in df_encoded.columns:
        target_means = df_encoded.groupby(col)['DEMENTED'].mean()
        df_encoded[f'{col}_encoded'] = df_encoded[col].map(target_means)
        df_encoded = df_encoded.drop(columns=[col])
        print(f"Target encoded: {col}")

numerical_to_scale = df_encoded.select_dtypes(include=[np.number]).columns
numerical_to_scale = [f for f in numerical_to_scale if f != 'DEMENTED']

scaler = StandardScaler()
df_encoded[numerical_to_scale] = scaler.fit_transform(df_encoded[numerical_to_scale])
print("Applied StandardScaler to numerical features")

print(f"After encoding and scaling: {df_encoded.shape}")

Low cardinality features: 3
High cardinality features: 1
Target encoded: NACCID
Applied StandardScaler to numerical features
After encoding and scaling: (195196, 93)


In [6]:
X = df_encoded.drop(columns=['DEMENTED'])
y = df_encoded['DEMENTED']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Testing target distribution:\n{y_test.value_counts(normalize=True)}")

Training set shape: (156156, 92)
Testing set shape: (39040, 92)
Training target distribution:
DEMENTED
0    0.704962
1    0.295038
Name: proportion, dtype: float64
Testing target distribution:
DEMENTED
0    0.704969
1    0.295031
Name: proportion, dtype: float64


In [7]:
processed_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'feature_names': X.columns.tolist(),
    'scaler': scaler
}

joblib.dump(processed_data, config.processed_data_dir / "feature_engineered_data.pkl")

X_train.to_csv(config.processed_data_dir / "X_train.csv", index=False)
X_test.to_csv(config.processed_data_dir / "X_test.csv", index=False)
y_train.to_csv(config.processed_data_dir / "y_train.csv", index=False)
y_test.to_csv(config.processed_data_dir / "y_test.csv", index=False)

feature_engineering_report = pd.DataFrame({
    'feature': X.columns,
    'feature_type': ['original' if not any(x in col for x in ['_score', '_index', '_interaction', '_group', '_level', '_squared', '_missing', '_encoded']) else 'engineered' for col in X.columns]
})
feature_engineering_report.to_csv(config.reports_dir / "feature_engineering_report.csv", index=False)

print("Feature engineering completed successfully!")
print(f"Final feature count: {X_train.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Files saved to: {config.processed_data_dir}")

Feature engineering completed successfully!
Final feature count: 92
Training samples: 156156
Testing samples: 39040
Files saved to: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\data\processed
